In [1]:
!pip install -q requests beautifulsoup4 lxml pandas nltk pyarabic qalsadi scikit-learn

import time, random, re, json
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Device: cuda


In [2]:
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer

ar_stop = set(stopwords.words("arabic"))
print(len(ar_stop), "stop words arabes")
print(ISRIStemmer().stem("المدرسة"))

701 stop words arabes
درس


In [3]:
import requests, time, re, os, random
import pandas as pd
from urllib.parse import urlparse
from urllib import robotparser
from bs4 import BeautifulSoup

OUT_DIR = "/kaggle/working/data"           
os.makedirs(OUT_DIR, exist_ok=True)
CSV_PATH = f"{OUT_DIR}/raw_articles.csv"

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; LSI-FSTT-LabBot/1.0; educational use)",
           "Accept-Language": "ar,en;q=0.8"}
session = requests.Session(); session.headers.update(HEADERS)

_robots = {}
def allowed(url):
    root = "{0.scheme}://{0.netloc}".format(urlparse(url))
    if root not in _robots:
        rp = None
        try:
            r = session.get(root + "/robots.txt", timeout=15)
            if r.status_code == 200:
                rp = robotparser.RobotFileParser(); rp.parse(r.text.splitlines())
            else:
                print(f"[robots] {root}: statut {r.status_code}, règles non lues")
        except requests.RequestException as e:
            print(f"[robots] {root}: {e}")
        _robots[root] = rp
    rp = _robots[root]
    return True if rp is None else rp.can_fetch(HEADERS["User-Agent"], url)

def get(url, retries=2):
    if not allowed(url):
        print("  bloqué par robots.txt :", url); return None
    for k in range(retries + 1):
        try:
            r = session.get(url, timeout=20); r.encoding = "utf-8"
            if r.status_code == 200: return r.text
            if r.status_code in (403, 429): time.sleep(5 * (k + 1))
        except requests.RequestException:
            time.sleep(2)
    return None

def clean_url(u): return u.split("#")[0].split("?")[0]

In [4]:
SOURCES = {
  "aljazeera": {
    "pattern": r"^https://www\.aljazeera\.net/[\w\-%]+/\d{4}/\d{1,2}/\d{1,2}/[^\s?#]+",
    "quota": 250,
    "seeds": [("https://www.aljazeera.net/tech/", "tech"),
              ("https://www.aljazeera.net/tag/ai/", "tech"),
              ("https://www.aljazeera.net/science/", "science"),
              ("https://www.aljazeera.net/ebusiness/", "economie"),
              ("https://www.aljazeera.net/health/", "sante"),
              ("https://www.aljazeera.net/culture/", "culture"),
              ("https://www.aljazeera.net/sport/", "sport"),
              ("https://www.aljazeera.net/politics/", "politique")],
  },
  "bbc": {
    "pattern": r"^https://www\.bbc\.com/arabic/articles/\w+",
    "quota": 125,
    "seeds": [("https://feeds.bbci.co.uk/arabic/rss.xml", "general"),                      # vérifié
              ("https://feeds.bbci.co.uk/arabic/science-and-tech/rss.xml", "tech"),        # à vérifier
              ("https://feeds.bbci.co.uk/arabic/middleeast/rss.xml", "general")],          # à vérifier
  },
  "dw": {
    "pattern": r"^https://www\.dw\.com/ar/[^\s?#]+/a-\d+",
    "quota": 125,
    "seeds": [("https://rss.dw.com/xml/rss-ar-all", "general"),                            # à vérifier
              ("https://www.dw.com/ar/", "general")],                                      # à vérifier
  },
}

def links_from_page(html, pattern, is_rss, base):
    soup = BeautifulSoup(html, "xml" if is_rss else "lxml")
    raw = ([i.find("link").get_text(strip=True) for i in soup.find_all("item") if i.find("link")]
           if is_rss else [a.get("href", "") for a in soup.find_all("a")])
    out = set()
    for h in raw:
        if h.startswith("/"): h = "https://" + urlparse(base).netloc + h
        h = clean_url(h)
        if re.match(pattern, h): out.add(h)
    return out

def discover(max_pages=6):
    found = {}                                   # url -> (source, section)
    for src, cfg in SOURCES.items():
        for seed, section in cfg["seeds"]:
            is_rss = "rss" in seed
            new_total = 0
            for page in range(1, 1 if is_rss else max_pages + 1):
                url = seed if page == 1 else f"{seed}?page={page}"
                html = get(url); time.sleep(random.uniform(1, 2))
                if not html: break
                links = links_from_page(html, cfg["pattern"], is_rss, seed) - set(found)
                if not links: break              # pagination ignorée ou fin atteinte
                for l in links: found[l] = (src, section)
                new_total += len(links)
            print(f"{src:10s} {section:9s} {new_total:4d} liens  <- {seed}")
    return found

found = discover()
print("\nTotal :", len(found), "| par source :", pd.Series([v[0] for v in found.values()]).value_counts().to_dict())

aljazeera  tech        41 liens  <- https://www.aljazeera.net/tech/
aljazeera  tech         4 liens  <- https://www.aljazeera.net/tag/ai/
aljazeera  science     31 liens  <- https://www.aljazeera.net/science/
aljazeera  economie    75 liens  <- https://www.aljazeera.net/ebusiness/
aljazeera  sante       75 liens  <- https://www.aljazeera.net/health/
aljazeera  culture     24 liens  <- https://www.aljazeera.net/culture/
aljazeera  sport       38 liens  <- https://www.aljazeera.net/sport/
aljazeera  politique   21 liens  <- https://www.aljazeera.net/politics/
bbc        general      0 liens  <- https://feeds.bbci.co.uk/arabic/rss.xml
bbc        tech         0 liens  <- https://feeds.bbci.co.uk/arabic/science-and-tech/rss.xml
bbc        general      0 liens  <- https://feeds.bbci.co.uk/arabic/middleeast/rss.xml
dw         general      0 liens  <- https://rss.dw.com/xml/rss-ar-all
dw         general     32 liens  <- https://www.dw.com/ar/

Total : 341 | par source : {'aljazeera': 309, 'dw'

In [5]:
from urllib.parse import unquote

SOURCES.pop("bbc", None)
SOURCES["aljazeera"]["quota"] = 300
SOURCES["dw"]["quota"] = 120
SOURCES["alaraby"] = {
    "pattern": r"^https://www\.alaraby\.co\.uk/(?!tag|author|topic|page|search|video)[\w\-]+/[^\s/?#]{12,}$",
    "quota": 120,
    "seeds": [("https://www.alaraby.co.uk/", "general"),
              ("https://www.alaraby.co.uk/economy", "economie"),
              ("https://www.alaraby.co.uk/entertainment_media", "medias"),
              ("https://www.alaraby.co.uk/technology", "tech")],          # à vérifier
}
SOURCES["aletihad"] = {
    "pattern": r"^https://www\.aletihad\.ae/news/[^/\s]+/\d+/[^\s?#]+",
    "quota": 80,
    "seeds": [("https://www.aletihad.ae/", "general")],
}

def discover(only=None, max_pages=6):
    found = {}
    for src, cfg in SOURCES.items():
        if only and src not in only: continue
        for seed, section in cfg["seeds"]:
            is_rss = "rss" in seed
            new_total = 0
            for page in range(1, 1 if is_rss else max_pages + 1):
                url = seed if page == 1 else f"{seed}?page={page}"
                html = get(url); time.sleep(random.uniform(1, 2))
                if not html: break
                links = links_from_page(html, cfg["pattern"], is_rss, seed) - set(found)
                if not links: break
                for l in links: found[l] = (src, section)
                new_total += len(links)
            print(f"{src:10s} {section:9s} {new_total:4d} liens  <- {seed}")
    return found

found = {u: v for u, v in found.items() if v[0] in SOURCES}      # retire BBC
found.update(discover(only=("alaraby", "aletihad")))
print("\nTotal :", len(found), "| par source :", pd.Series([v[0] for v in found.values()]).value_counts().to_dict())

[robots] https://www.alaraby.co.uk: statut 403, règles non lues
alaraby    general      0 liens  <- https://www.alaraby.co.uk/
alaraby    economie     0 liens  <- https://www.alaraby.co.uk/economy
alaraby    medias       0 liens  <- https://www.alaraby.co.uk/entertainment_media
alaraby    tech         0 liens  <- https://www.alaraby.co.uk/technology
aletihad   general    136 liens  <- https://www.aletihad.ae/

Total : 477 | par source : {'aljazeera': 309, 'aletihad': 136, 'dw': 32}


In [6]:
SOURCES.pop("alaraby", None)                 # bloqué (403)
SOURCES["aletihad"]["quota"] = 150           # il fournit plus de liens que prévu
found = {u: v for u, v in found.items() if v[0] in SOURCES}
print(len(found), pd.Series([v[0] for v in found.values()]).value_counts().to_dict())
print({s: SOURCES[s]["quota"] for s in SOURCES})

477 {'aljazeera': 309, 'aletihad': 136, 'dw': 32}
{'aljazeera': 300, 'dw': 120, 'aletihad': 150}


In [7]:
from collections import deque

TECH_CAP = 170

def extract(html):
    soup = BeautifulSoup(html, "lxml")
    og = soup.find("meta", property="og:title")
    title = og.get("content", "") if og else (soup.title.get_text(strip=True) if soup.title else "")
    dt = soup.find("meta", property="article:published_time")
    date = dt.get("content", "") if dt else ""
    for t in soup(["script", "style", "nav", "footer", "aside", "form", "figure", "noscript"]):
        t.decompose()
    root = soup.find("article") or soup.find("main") or soup.body
    paras = [p.get_text(" ", strip=True) for p in root.find_all("p")] if root else []
    paras = [p for p in paras if len(p) > 40 and re.search(r"[\u0600-\u06FF]", p)]
    return title.strip(), date, "\n".join(paras)

key = lambda u: unquote(u).rstrip("/")          # évite les doublons dus à l'encodage de l'URL

def section_of(url, src, default):
    if src == "aljazeera":
        m = re.match(r"https://www\.aljazeera\.net/([^/]+)/", url)
        return unquote(m.group(1)) if m else default
    return default

buffer = []
def flush():
    global buffer
    if buffer:
        pd.DataFrame(buffer).to_csv(CSV_PATH, mode="a", header=not os.path.exists(CSV_PATH),
                                    index=False, encoding="utf-8-sig")
        buffer = []

count = {s: 0 for s in SOURCES}; n_tech = 0; seen = set()
if os.path.exists(CSV_PATH):
    old = pd.read_csv(CSV_PATH)
    seen = {key(u) for u in old["url"]}
    for s, n in old["source"].value_counts().items():
        if s in count: count[s] = int(n)
    n_tech = int(((old["source"] == "aljazeera") & (old["section"] == "tech")).sum())

queue = []
for u, (s, sec) in found.items():
    if key(u) not in seen:
        seen.add(key(u)); queue.append((u, s, section_of(u, s, sec)))
random.shuffle(queue)
queue.sort(key=lambda t: not (t[1] == "aljazeera" and t[2] == "tech"))     # technologie d'abord
queue = deque(queue)

n_proc = 0
while queue and any(count[s] < SOURCES[s]["quota"] for s in SOURCES):
    url, src, section = queue.popleft()
    if count[src] >= SOURCES[src]["quota"]: continue
    is_tech = (src == "aljazeera" and section == "tech")
    if is_tech and n_tech >= TECH_CAP: continue
    html = get(url); time.sleep(random.uniform(1, 2))
    if not html: continue
    for l in links_from_page(html, SOURCES[src]["pattern"], False, url):     # articles liés
        if key(l) not in seen:
            seen.add(key(l))
            sec = section_of(l, src, "related")
            (queue.appendleft if (src == "aljazeera" and sec == "tech") else queue.append)((l, src, sec))
    title, date, text = extract(html)
    if len(text) < 300: continue                       # vidéo, galerie ou page vide
    buffer.append({"url": url, "source": src, "section": section,
                   "title": title, "date": date, "text": text})
    count[src] += 1; n_tech += int(is_tech); n_proc += 1
    if len(buffer) >= 25:
        flush(); print(f"{n_proc} articles | {count} | tech Al Jazeera : {n_tech} | file : {len(queue)}")
flush()
df = pd.read_csv(CSV_PATH)
print("\nTerminé :", len(df), "articles |", df["source"].value_counts().to_dict())

25 articles | {'aljazeera': 25, 'dw': 0, 'aletihad': 0} | tech Al Jazeera : 25 | file : 544
50 articles | {'aljazeera': 50, 'dw': 0, 'aletihad': 0} | tech Al Jazeera : 50 | file : 612
75 articles | {'aljazeera': 75, 'dw': 0, 'aletihad': 0} | tech Al Jazeera : 75 | file : 659
100 articles | {'aljazeera': 100, 'dw': 0, 'aletihad': 0} | tech Al Jazeera : 100 | file : 696
125 articles | {'aljazeera': 125, 'dw': 0, 'aletihad': 0} | tech Al Jazeera : 125 | file : 722
150 articles | {'aljazeera': 150, 'dw': 0, 'aletihad': 0} | tech Al Jazeera : 150 | file : 741
175 articles | {'aljazeera': 172, 'dw': 0, 'aletihad': 3} | tech Al Jazeera : 170 | file : 657
200 articles | {'aljazeera': 192, 'dw': 0, 'aletihad': 8} | tech Al Jazeera : 170 | file : 765
225 articles | {'aljazeera': 204, 'dw': 3, 'aletihad': 18} | tech Al Jazeera : 170 | file : 837
250 articles | {'aljazeera': 222, 'dw': 5, 'aletihad': 23} | tech Al Jazeera : 170 | file : 915
275 articles | {'aljazeera': 235, 'dw': 8, 'aletihad': 32

In [8]:
df = pd.read_csv(CSV_PATH)
df["k"] = df["url"].map(key)
df = df.drop_duplicates("k").drop_duplicates("title").drop(columns="k")
df["n_chars"] = df["text"].str.len()
df.to_csv(f"{OUT_DIR}/articles_dedup.csv", index=False, encoding="utf-8-sig")

print(len(df), "articles uniques")
print(df.groupby(["source", "section"]).size().sort_values(ascending=False).head(25))
print(df["n_chars"].describe().round(0))
print(df.sample(1, random_state=1)[["source", "title"]].to_string())
print(df.sample(1, random_state=1)["text"].iloc[0][:500])

570 articles uniques
source     section  
aljazeera  tech         170
aletihad   general      118
dw         related       92
aljazeera  health        37
           ebusiness     34
aletihad   related       32
dw         general       28
aljazeera  sport         13
           science       12
           culture       12
           politics      11
           news           4
           opinions       3
           video          2
           misc           1
           lifestyle      1
dtype: int64
count      570.0
mean      3087.0
std       2676.0
min        304.0
25%       1299.0
50%       2310.0
75%       3972.0
max      22818.0
Name: n_chars, dtype: float64
    source                                            title
565     dw  حين يصبح المناخ قضية دستورية.. أين تقف ألمانيا؟
تلتزم الدول الأطراف في اتفاقية باريس بشأن تغير المناخ بمكافحة الاحتباس الحراري. لكن بعض الدول تذهب أبعد من ذلك، فتكرّس حماية المناخ في دساتيرها. فما الذي يميز ألمانيا عن الإكوادور وفيتنام وتوفالو؟
بتوقيع اتفاقية

In [14]:
from collections import Counter
df = pd.read_csv(f"{OUT_DIR}/articles_dedup.csv")

TASHKEEL = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0640]")
def norm_ar(t):
    t = TASHKEEL.sub("", str(t))
    t = re.sub("[إأآٱ]", "ا", t).replace("ى", "ي")
    return t.lower()

AR = "\u0621-\u064A"
PRE = r"[وف]?(?:لل|[بكل]?(?:ال)?)"
SUF = r"(?:ي?ات|ي?ة|ان|ين|ون|ي|يا)?"
def P(term): return re.compile(rf"(?<![{AR}]){PRE}(?:{term}){SUF}(?![{AR}])")

LEX = []
def add(w, *terms): LEX.extend((w, P(t)) for t in terms)

add(3,   r"ذكاء\s+(?:ال)?اصطناعي", r"تعلم\s+(?:ال)?الي", r"تعلم\s+(?:ال)?عميق",
         r"شبكات?\s+(?:ال)?عصبي", r"(?:نموذج|نماذج)\s+(?:ال)?لغوي",
         r"روبوت\w*\s+(?:ال)?(?:دردشة|محادثة)", r"معالجة\s+(?:ال)?لغات", r"ت?شات\s*جي\s*بي\s*تي")
LEX.append((3, re.compile(r"\b(?:ai|chatgpt|openai|llm|gpt-?\d*|deepseek|copilot)\b")))
add(1.5, r"روبوت", r"خوارزمي", r"حوسب", r"سيبراني", r"برمج", r"رقاق", r"حاسوب", r"كمبيوتر",
         r"انفيديا", r"تقني", r"تكنولوج", r"سحابي", r"بيانات\s+(?:ال)?ضخمة",
         r"شرائح\s+(?:ال)?(?:ذكية|الكترونية)",
         r"متصفح", r"ويندوز", r"لينوكس", r"اندرويد", r"ايفون", r"غوغل", r"جوجل", r"مايكروسوفت",
         r"العاب\s+(?:ال)?(?:الكترونية|فيديو)")
add(0.5, r"رقمي", r"الكترون", r"انترنت", r"هاتف")       # retirés : تطبيق، ابتكار، مبتكر (trop génériques)

def score_parts(title, text):
    t, ti = norm_ar(text), norm_ar(title)
    n_strong = sum(len(p.findall(t)) for w, p in LEX if w >= 1.5)
    strong = sum(w * len(p.findall(t)) for w, p in LEX if w >= 1.5)
    weak = min(sum(len(p.findall(t)) for w, p in LEX if w < 1.5), 4) * 0.5
    density = 1000 * (strong + weak) / max(len(t.split()), 200)
    core = sum(len(p.findall(ti)) for w, p in LEX if w == 3)
    other = sum(len(p.findall(ti)) for w, p in LEX if w < 3)
    # --- indices de "vrai sujet tech/IA" ---
    core_text = sum(len(p.findall(t)) for w, p in LEX if w == 3)                # occurrences IA/ML/ChatGPT...
    distinct = sum(1 for w, p in LEX if w >= 1.5 and p.search(t))               # termes forts différents
    title_hit = sum(len(p.findall(ti)) for w, p in LEX if w >= 1.5) > 0         # terme fort dans le titre
    raw = density + 8 * (core > 0) + 3 * (other > 0)
    return raw, n_strong, core_text, distinct, title_hit

parts = [score_parts(a, b) for a, b in zip(df["title"], df["text"])]
df["raw"]       = [p[0] for p in parts]
df["n_strong"]  = [p[1] for p in parts]
df["core_text"] = [p[2] for p in parts]
df["distinct"]  = [p[3] for p in parts]
df["title_hit"] = [p[4] for p in parts]

m = max(df.loc[(df.source == "aljazeera") & (df.section == "tech"), "raw"].median(), 1.0)
s = m / np.log(4)
print(f"médiane brute tech = {m:.1f} | échelle s = {s:.1f}")
df["score"] = (10 * (1 - np.exp(-df["raw"] / s)) * 2).round() / 2

# plafond à 3 sauf preuve que l'article parle vraiment de tech/IA
evidence = df["title_hit"] | (df["core_text"] >= 3) | (df["distinct"] >= 4)
df["score"] = np.where(evidence, df["score"], df["score"].clip(upper=3))
print(pd.cut(df["score"], [-0.1, 2, 4, 6, 8, 10.1]).value_counts().sort_index())

# --- Diagnostic : pourquoi ces articles non-tech ont-ils un score >= 6 ? ---
def explain(text, top=4):
    t, c = norm_ar(text), Counter()
    for w, p in LEX:
        for mt in p.finditer(t): c[(w, mt.group(0))] += 1
    return [(f"{w}:{x}", n) for (w, x), n in c.most_common(top)]

sus = df[(df.score >= 6) & ~((df.source == "aljazeera") & (df.section == "tech"))].sort_values("score", ascending=False)
print(f"\n{len(sus)} articles non-tech avec score >= 6")
for _, r in sus.head(12).iterrows():
    print(f"{r.score:4.1f} | {r.source:9s} | {str(r.title)[:60]} | {len(r.text.split())} mots")
    print("      ", explain(r.text))

low = df[(df.source == "aljazeera") & (df.section == "tech") & (df.score <= 3)]
print(f"\n{len(low)} articles tech d'Al Jazeera avec score <= 3")
for _, r in low.head(10).iterrows(): print(f"{r.score:4.1f} | {str(r.title)[:80]}")

médiane brute tech = 35.5 | échelle s = 25.6
score
(-0.1, 2.0]    337
(2.0, 4.0]     102
(4.0, 6.0]      16
(6.0, 8.0]      36
(8.0, 10.1]     79
Name: count, dtype: int64

32 articles non-tech avec score >= 6
10.0 | aljazeera | 4.9 تريليونات دولار إيرادات الصناعة الرقمية في الصين | 204 mots
       [('3:الذكاء الاصطناعي', 5), ('0.5:الرقمية', 4), ('1.5:الحوسبة', 2), ('3:بالذكاء الاصطناعي', 1)]
10.0 | aljazeera | تقرير للكونغرس: الجيش الصيني يطارد رقائق إنفيديا من أجل "الح | 341 mots
       [('3:الذكاء الاصطناعي', 5), ('1.5:انفيديا', 5), ('3:بالذكاء الاصطناعي', 2), ('1.5:الحوسبة', 1)]
10.0 | dw        | DW تتحقق: كيف يمكنني التعرف على الصور المبتكرة بواسطة الذكاء | 1080 mots
       [('3:الذكاء الاصطناعي', 22), ('3:بالذكاء الاصطناعي', 3), ('3:openai', 2), ('1.5:التكنولوجيا', 2)]
10.0 | aletihad  | الدولة: تعزيز التعاون الدولي لمساعدة الدول على تطوير أنظمة ذ | 143 mots
       [('3:الذكاء الاصطناعي', 4), ('3:ذكاء اصطناعي', 2)]
10.0 | dw        | الذكاء الاصطناعي في المدارس: ثورة تعليمية أم 

In [15]:
checks = ["اختبارات نهاية الفصل", "أبوظبي الأول", "سلامة الأغذية", "موسم التخفيضات",
          "القمة العالمية للحكومات", "تطبيق للقرآن", "الألعاب الإلكترونية في دعم", "سافاري"]
for c in checks:
    for _, r in df[df["title"].str.contains(c, na=False)].head(2).iterrows():
        print(f"{r.score:4.1f} | {r.source:9s} | {str(r.title)[:70]}")

 3.0 | aletihad  | "التربية" تعلن البرنامج الزمني لاختبارات نهاية الفصل الدراسي الأول
 3.0 | aletihad  | «أبوظبي الأول» يوافق على مقترح إصدار حقوق أولوية بـ 15 مليار درهم
 3.0 | aletihad  | «دبي العالمي لسلامة الأغذية» يناقش مستقبل الأنظمة الغذائية
 3.0 | dw        | احذر التسوق في موسم التخفيضات لهذه الأسباب
 6.0 | aletihad  | إطلاق شراكات استراتيجية ومبادرات في حوار القمة العالمية للحكومات لأمير
 1.5 | aletihad  | ضمن حوار القمة العالمية للحكومات فـي الدومينيكان.. رؤساء وقادة حكومات 
 9.0 | aljazeera | تطبيق للقرآن الكريم على أندرويد وآيفون
 5.0 | aljazeera | آثار كبيرة.. دور الألعاب الإلكترونية في دعم الاقتصاد
 6.5 | aljazeera | نسخة ثلاثية الأبعاد من متصفح سافاري


In [21]:
print(df.groupby(["source", "section"])["score"].agg(["count", "mean", "median"])
        .round(1).sort_values("mean", ascending=False).head(15))
print("\nRépartition des scores :")
print(pd.cut(df["score"], [-0.1, 2, 4, 6, 8, 10.1]).value_counts().sort_index())

val = df.sample(30, random_state=7).sort_values("score")
print("\nÉchantillon à relire :")
for _, r in val.iterrows():
    print(f"{r.score:4.1f} | {r.source:9s} | {str(r.title)[:75]}")
val[["source", "title", "score"]].assign(manual_score="").to_csv(
    f"{OUT_DIR}/validation_sample.csv", index=False, encoding="utf-8-sig")

                     count  mean  median
source    section                       
aljazeera tech         170   5.8     6.2
          politics      11   3.2     0.0
          ebusiness     34   2.8     1.5
          news           4   2.5     0.0
dw        general       28   1.9     0.8
aljazeera science       12   1.6     1.0
dw        related       92   1.3     0.0
aletihad  general      118   0.9     0.0
aljazeera opinions       3   0.7     0.5
          health        37   0.4     0.0
aletihad  related       32   0.3     0.0
aljazeera culture       12   0.2     0.0
          sport         13   0.2     0.0
          lifestyle      1   0.0     0.0
          misc           1   0.0     0.0

Répartition des scores :
score
(-0.1, 2.0]    337
(2.0, 4.0]     102
(4.0, 6.0]      16
(6.0, 8.0]      36
(8.0, 10.1]     79
Name: count, dtype: int64

Échantillon à relire :
 0.0 | aletihad  | نخيل: «بالم سبرينغز» يسجّل مبيعات بقيمة 5.5 مليار درهم في مرحلته الأولى
 0.0 | aletihad  | تحديد عطلة الحكو

In [22]:
df[["text", "score"]].to_csv(f"{OUT_DIR}/dataset.csv", index=False, encoding="utf-8-sig")   # format de l'énoncé
df.to_csv(f"{OUT_DIR}/articles_scored.csv", index=False, encoding="utf-8-sig")              # version complète
print(len(df), "lignes enregistrées")

570 lignes enregistrées
